In [27]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import classification_report, confusion_matrix
import joblib

MODEL_FEATURE_COLUMNS = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
    'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
    'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total',
    'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length',
    'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Std', 'FIN Flag Count', 'RST Flag Count',
    'PSH Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward',
    'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward',
    'Active Max', 'Idle Mean',
]

df = pd.read_csv('../Preprocessing/combined.csv')
X = df[MODEL_FEATURE_COLUMNS].copy()

# MODEL_FEATURE_COLUMNS = list(COLUMN_MAP.values())  # preserves the order above

# --- Load artifacts ---
model = joblib.load("../newOutput/ensemble_model.pkl")
scaler = joblib.load("../newOutput/scaler.pkl")
label_encoder = joblib.load("../newOutput/label_encoder.pkl")

# --- Load CSV ---
# df = pd.read_csv("../dataset/IDS2018/02-15-2018/02-15-2018.csv")
df = pd.read_csv('../Preprocessing/combined.csv')

# Rename only the columns we need
# df = df.rename(columns=COLUMN_MAP)

# Select and order the 40 features
X = df[MODEL_FEATURE_COLUMNS].copy()
# Clean: coerce to numeric, replace inf/-inf, fill NaN with 0
X = X.apply(pd.to_numeric, errors="coerce")
X.replace([np.inf, -np.inf], 0, inplace=True)
X.fillna(0, inplace=True)

# Scale
X_scaled = scaler.transform(X.values)

# Predict
y_pred_enc = model.predict(X_scaled)
y_pred = label_encoder.inverse_transform(y_pred_enc)

# Ground truth - adjust the label column name if it differs
y_true = df["Attack Type"].values


c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


In [24]:
print(df.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Max', 'Idle Mean', 'Attack Type', 'Source']


In [25]:
COLUMN_MAP_2 = {
    "Destination Port":          "Destination Port",  # not in model 40, will be dropped
    "Flow Duration":             "Flow Duration",
    "Total Fwd Packets":         "Total Fwd Packets",
    "Total Backward Packets":    "Total Backward Packets",
    "Total Length of Fwd Packets": "Total Length of Fwd Packets",
    "Total Length of Bwd Packets": "Total Length of Bwd Packets",  # not in model 40, dropped
    "Fwd Packet Length Max":     "Fwd Packet Length Max",
    "Fwd Packet Length Min":     "Fwd Packet Length Min",
    "Fwd Packet Length Mean":    "Fwd Packet Length Mean",
    "Fwd Packet Length Std":     "Fwd Packet Length Std",
    "Bwd Packet Length Max":     "Bwd Packet Length Max",
    "Bwd Packet Length Min":     "Bwd Packet Length Min",
    "Bwd Packet Length Mean":    "Bwd Packet Length Mean",
    "Bwd Packet Length Std":     "Bwd Packet Length Std",
    "Flow Bytes/s":              "Flow Bytes/s",
    "Flow Packets/s":            "Flow Packets/s",
    "Flow IAT Mean":             "Flow IAT Mean",
    "Flow IAT Std":              "Flow IAT Std",
    "Flow IAT Max":              "Flow IAT Max",
    "Flow IAT Min":              "Flow IAT Min",
    "Fwd IAT Mean":              None,  # not in model 40, dropped
    "Fwd IAT Std":               "Fwd IAT Std",
    "Fwd IAT Min":               "Fwd IAT Min",
    "Bwd IAT Total":             "Bwd IAT Total",
    "Bwd IAT Mean":              None,  # not in model 40, dropped
    "Bwd IAT Max":               "Bwd IAT Max",
    "Bwd IAT Min":               "Bwd IAT Min",
    "Fwd Header Length":         "Fwd Header Length",
    "Bwd Header Length":         "Bwd Header Length",
    "Bwd Packets/s":             "Bwd Packets/s",
    "Min Packet Length":         "Min Packet Length",
    "Max Packet Length":         "Max Packet Length",
    "Packet Length Mean":        "Packet Length Mean",
    "Packet Length Std":         "Packet Length Std",
    "Packet Length Variance":    None,  # not in model 40, dropped
    "FIN Flag Count":            "FIN Flag Count",
    "PSH Flag Count":            "PSH Flag Count",
    "ACK Flag Count":            "ACK Flag Count",
    "Init_Win_bytes_forward":    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward":   "Init_Win_bytes_backward",
    "act_data_pkt_fwd":          "act_data_pkt_fwd",
    "min_seg_size_forward":      "min_seg_size_forward",
    "Active Mean":               "Active Mean",
    "Active Max":                "Active Max",
    "Active Min":                "Active Min",
    "Idle Mean":                 "Idle Mean",
}

LABEL_MAP_2 = {
    "Port Scan": "Port Scan",
}

df2 = pd.read_csv("../PortScan.csv")

# Drop columns with no model mapping
drop_cols = [k for k, v in COLUMN_MAP_2.items() if v is None]
df2 = df2.drop(columns=drop_cols)

X2 = df2[MODEL_FEATURE_COLUMNS].copy()

X2 = X2.apply(pd.to_numeric, errors="coerce")
X2.replace([np.inf, -np.inf], 0, inplace=True)
X2.fillna(0, inplace=True)

X2_scaled = scaler.transform(X2.values)

y2_pred_enc = model.predict(X2_scaled)
y2_pred = label_encoder.inverse_transform(y2_pred_enc)

y2_true_mapped = pd.Series(df2["Attack Type"]).map(LABEL_MAP_2).values

print(classification_report(y2_true_mapped, y2_pred))

present_labels2 = sorted(set(y2_true_mapped))
cm2 = pd.DataFrame(
    confusion_matrix(y2_true_mapped, y2_pred, labels=present_labels2),
    index=present_labels2,
    columns=present_labels2
)
print(cm2)

results2 = df2[["Attack Type"]].copy()
results2["Mapped Label"] = y2_true_mapped
results2["Predicted"] = y2_pred
results2["Correct"] = results2["Mapped Label"] == results2["Predicted"]
print(f"\nOverall Accuracy: {results2['Correct'].mean():.4f}")

c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                precision    recall  f1-score   support

           DoS       0.00      0.00      0.00         0
Normal Traffic       0.00      0.00      0.00         0
     Port Scan       1.00      1.00      1.00     90694

      accuracy                           1.00     90694
     macro avg       0.33      0.33      0.33     90694
  weighted avg       1.00      1.00      1.00     90694

           Port Scan
Port Scan      90680

Overall Accuracy: 0.9998


c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\abhinav\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [31]:
y_true = df["Attack Type"].values
present_labels = sorted(set(y_true))

print(classification_report(y_true, y_pred, labels=present_labels, zero_division=0))

cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=present_labels),
    index=present_labels,
    columns=present_labels
)
print(cm)

results = df[["Attack Type"]].copy()
results["Predicted"] = y_pred
results["Correct"] = results["Attack Type"] == results["Predicted"]
print(f"\nOverall Accuracy: {results['Correct'].mean():.4f}")

                precision    recall  f1-score   support

   Brute Force       1.00      1.00      1.00    103244
          DDoS       1.00      1.00      1.00    328605
           DoS       1.00      1.00      1.00    390200
Normal Traffic       1.00      1.00      1.00    887162
     Port Scan       1.00      1.00      1.00     90694

      accuracy                           1.00   1799905
     macro avg       1.00      1.00      1.00   1799905
  weighted avg       1.00      1.00      1.00   1799905

                Brute Force    DDoS     DoS  Normal Traffic  Port Scan
Brute Force          103242       0       2               0          0
DDoS                      0  328598       0               7          0
DoS                      38       0  390072              75         15
Normal Traffic           13      27     287          886777         58
Port Scan                 0       0       4               4      90686

Overall Accuracy: 0.9997


In [12]:
print("Label encoder classes:", label_encoder.classes_)
print("CSV labels:", pd.Series(y_true).unique())

Label encoder classes: ['Brute Force' 'DDoS' 'DoS' 'Normal Traffic' 'Port Scan']
CSV labels: <StringArray>
['Benign', 'DoS attacks-GoldenEye', 'DoS attacks-Slowloris']
Length: 3, dtype: str
